<a href="https://colab.research.google.com/github/ss49880-a11y/ProjectJackfruit101/blob/main/train_jackfruit_image_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ProjectJackfruit101 — Image CNN Training
จำแนกความสุกขนุน 2 คลาส: ขนุนดิบ / ขนุนสุก

## Step 1 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# ชื่อ folder ต้องตรงกับใน Drive เป๊ะๆ
DATASET_PATH = '/content/drive/MyDrive'
CLASSES      = ['ขนุนดิบ', 'ขนุนสุก']
OUTPUT_PATH  = '/content/drive/MyDrive/jackfruit_image_model'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 2 — Import

In [2]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

print('TensorFlow version:', tf.__version__)

TensorFlow version: 2.20.0


## Step 3 — White Balance Correction

In [3]:
def white_balance(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).astype(np.float32)
    l, a, b = cv2.split(lab)
    a = np.clip(a - (np.mean(a) - 128), 0, 255)
    b = np.clip(b - (np.mean(b) - 128), 0, 255)
    balanced = cv2.merge([l, a, b]).astype(np.uint8)
    return cv2.cvtColor(balanced, cv2.COLOR_LAB2BGR)

## Step 4 — โหลด Dataset + White Balance

In [4]:
IMG_SIZE = 224
images, labels = [], []

for idx, class_name in enumerate(CLASSES):
    class_path = os.path.join(DATASET_PATH, class_name)
    files = os.listdir(class_path)
    print(f'{class_name}: {len(files)} รูป')

    for fname in files:
        fpath = os.path.join(class_path, fname)
        img = cv2.imread(fpath)
        if img is None:
            continue
        img = white_balance(img)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img / 255.0
        images.append(img)
        labels.append(idx)

images = np.array(images, dtype=np.float32)
labels = np.array(labels)
print(f'\nโหลดทั้งหมด: {len(images)} รูป')

ขนุนดิบ: 291 รูป
ขนุนสุก: 306 รูป

โหลดทั้งหมด: 597 รูป


## Step 5 — Split 70/15/15

In [5]:
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.30, stratify=labels, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)
print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

Train: 417 | Val: 90 | Test: 90


## Step 6 — Data Augmentation

In [6]:
datagen = ImageDataGenerator(
    rotation_range=30,
    horizontal_flip=True,
    brightness_range=[0.4, 1.4],
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1
)

y_train_oh = tf.keras.utils.to_categorical(y_train, num_classes=2)
y_val_oh   = tf.keras.utils.to_categorical(y_val,   num_classes=2)
y_test_oh  = tf.keras.utils.to_categorical(y_test,  num_classes=2)

train_gen = datagen.flow(X_train, y_train_oh, batch_size=32)
print('Augmentation พร้อมแล้ว')

Augmentation พร้อมแล้ว


## Step 7 — สร้าง Model (MobileNetV2)

In [7]:
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = base_model(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.Dense(128, activation='relu')(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(2, activation='softmax')(x)

model = Model(inputs, outputs)
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,210 (9.24 MB)

 Trainable params: 164,226 (641.51 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## Step 8 — Train Model

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

history = model.fit(
    train_gen,
    validation_data=(X_val, y_val_oh),
    epochs=50,
    callbacks=[early_stop]
)
print('Training เสร็จแล้ว')

Epoch 1/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 36s 2s/step - accuracy: 0.5180 - loss: 0.9312 - val_accuracy: 0.3333 - val_loss: 0.8865
Epoch 2/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 26s 2s/step - accuracy: 0.5204 - loss: 0.8393 - val_accuracy: 0.3444 - val_loss: 0.8757
Epoch 3/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 26s 2s/step - accuracy: 0.5516 - loss: 0.8005 - val_accuracy: 0.4556 - val_loss: 1.0996
Epoch 4/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 29s 2s/step - accuracy: 0.4892 - loss: 0.7990 - val_accuracy: 0.4889 - val_loss: 1.2084
Epoch 5/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 40s 2s/step - accuracy: 0.5132 - loss: 0.7362 - val_accuracy: 0.4222 - val_loss: 0.8068
Epoch 6/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 30s 2s/step - accuracy: 0.5084 - loss: 0.7086 - val_accuracy: 0.4778 - val_loss: 0.8741
Epoch 7/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 27s 2s/step - accuracy: 0.5036 - loss: 0.7098 - val_accuracy: 0.4889 - val_loss: 0.9791
Epoch 8/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 28s 2s/step - accuracy: 0.5132 - loss: 0.6853 - val_accuracy: 0.4778 - val_loss:

## Step 9 — ประเมินผล

In [ ]:
loss, acc = model.evaluate(X_test, y_test_oh)
print(f'Test Accuracy: {acc*100:.2f}%')

y_pred = np.argmax(model.predict(X_test), axis=1)
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=CLASSES))

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title('Accuracy')
plt.legend()
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('Loss')
plt.legend()
plt.tight_layout()
plt.show()

## Step 10 — แปลงเป็น TFLite

In [ ]:
os.makedirs(OUTPUT_PATH, exist_ok=True)

converter    = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

tflite_path = os.path.join(OUTPUT_PATH, 'jackfruit_image.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f'บันทึก TFLite ที่: {tflite_path}')
print(f'ขนาดไฟล์: {os.path.getsize(tflite_path)/1024:.1f} KB')

## Step 11 — ทดสอบ TFLite

In [ ]:
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

test_img = np.expand_dims(X_test[0], axis=0).astype(np.float32)
interpreter.set_tensor(input_details[0]['index'], test_img)
interpreter.invoke()

result = interpreter.get_tensor(output_details[0]['index'])[0]
pred_class = CLASSES[np.argmax(result)]
print(f'ผลทำนาย: {pred_class}')
for i, c in enumerate(CLASSES):
    print(f'  {c}: {result[i]*100:.1f}%')